# Customer, Product, and Profitability Performance Analysis in Supply Chain Operations

**APL Logistics (KWE Group)** | Unified Mentor Project

---

Leadership question: *Which customers, products, and regions truly generate value for the business?*

This notebook performs the full analytical methodology requested:

1. Data Cleaning & Financial Validation
2. Revenue & Profit Overview
3. Product & Category Profitability Analysis
4. Customer Contribution Analysis
5. Discount Impact Diagnostics
6. Market & Regional Profit Analysis
7. KPI Summary
8. Key Insights & Recommendations

The cleaned output of this notebook (`apl_clean.parquet` / `apl_clean.csv`) feeds the companion Streamlit dashboard (`dashboard/app.py`).


## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path

pio.templates.default = "plotly_white"
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# Fintech blue palette used consistently across all charts
PALETTE = ["#0B3D91", "#1D5FBF", "#3E8DED", "#7FB6F7", "#B7D8FB", "#0A2540"]
NEG_COLOR = "#D64545"


In [2]:
DATA_PATH = Path('../data/APL_Logistics.csv')
df_raw = pd.read_csv(DATA_PATH, encoding='latin1')
print(f"Rows: {df_raw.shape[0]:,}  |  Columns: {df_raw.shape[1]}")
df_raw.head()


Rows: 180,519  |  Columns: 40


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Fname,Customer Id,Customer Lname,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,Order Item Discount,Order Item Discount Rate,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Product Name,Product Price,Shipping Mode
0,DEBIT,6,4,159.69,472.45,Late delivery,1,9,Cardio Equipment,Brownsville,EE. UU.,Richard,1,Hernandez,Consumer,TX,6303 Heather Plaza,"78,521.00",3,Footwear,25.95,-97.51,Pacific Asia,Mumbai,India,1,27.50,0.06,99.99,0.34,5,499.95,472.45,159.69,South Asia,Maharashtra,COMPLETE,Nike Men's Free 5.0+ Running Shoe,99.99,Standard Class
1,DEBIT,4,4,48.71,167.96,Shipping on time,0,29,Shop By Sport,Littleton,EE. UU.,Mary,2,Barrett,Consumer,CO,9526 Noble Embers Ridge,"80,126.00",5,Golf,38.38,-104.73,LATAM,San Pedro Sula,Honduras,2,31.99,0.16,39.99,0.29,5,199.95,167.96,48.71,Central America,Cortés,ON_HOLD,Under Armour Girls' Toddler Spine Surge Runni,39.99,Standard Class
2,DEBIT,4,4,87.36,181.99,Shipping on time,0,48,Water Sports,Littleton,EE. UU.,Mary,2,Barrett,Consumer,CO,9526 Noble Embers Ridge,"80,126.00",7,Fan Shop,38.38,-104.73,LATAM,San Pedro Sula,Honduras,2,18.00,0.09,199.99,0.48,1,199.99,181.99,87.36,Central America,Cortés,ON_HOLD,Pelican Sunstream 100 Kayak,199.99,Standard Class
3,DEBIT,6,4,-41.89,175.99,Late delivery,1,48,Water Sports,Littleton,EE. UU.,Mary,2,Barrett,Consumer,CO,9526 Noble Embers Ridge,"80,126.00",7,Fan Shop,38.38,-104.73,USCA,New York City,Estados Unidos,2,24.00,0.12,199.99,-0.24,1,199.99,175.99,-41.89,East of USA,Nueva York,COMPLETE,Pelican Sunstream 100 Kayak,199.99,Standard Class
4,DEBIT,6,4,10.00,40.00,Late delivery,1,24,Women's Apparel,Littleton,EE. UU.,Mary,2,Barrett,Consumer,CO,9526 Noble Embers Ridge,"80,126.00",5,Golf,38.38,-104.73,USCA,New York City,Estados Unidos,2,10.00,0.20,50.00,0.25,1,50.00,40.00,10.00,East of USA,Nueva York,COMPLETE,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class


## 2. Data Cleaning & Financial Validation

Steps:
- Inspect nulls, duplicates, and data types
- Validate that financial fields (Sales, Benefit per order, Order Profit Per Order) are numeric and sensible
- Remove zero/invalid-value order records
- Normalize/derive a consistent `Profit Margin` field


In [3]:
# Null / duplicate audit
null_counts = df_raw.isnull().sum()
print("Columns with nulls:")
print(null_counts[null_counts > 0])
print(f"\nDuplicate rows: {df_raw.duplicated().sum()}")


Columns with nulls:
Customer Lname      8
Customer Zipcode    3
dtype: int64



Duplicate rows: 0


In [4]:
df = df_raw.copy()

# Drop exact duplicate order-item rows, if any
df = df.drop_duplicates()

# Financial validation: Sales and Order Item Total must be > 0 to represent a real transaction
before = len(df)
df = df[(df['Sales'] > 0) & (df['Order Item Total'] > 0) & (df['Order Item Quantity'] > 0)]
after = len(df)
print(f"Removed {before - after} zero/invalid-value order records ({before-after} rows)")

# Fill missing Customer Lname / Zipcode - not used in profitability math, keep as 'Unknown'/0
df['Customer Lname'] = df['Customer Lname'].fillna('Unknown')
df['Customer Zipcode'] = df['Customer Zipcode'].fillna(0)

# Derived, normalized profitability field (source of truth used throughout the notebook)
df['Profit Margin %'] = np.where(df['Sales'] != 0, (df['Benefit per order'] / df['Sales']) * 100, 0)

# Sanity clip: cap extreme outlier margins for visualization stability only (does not touch KPI totals)
df['Profit Margin % (clipped)'] = df['Profit Margin %'].clip(-100, 100)

print(f"\nClean dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
df[['Sales', 'Benefit per order', 'Order Profit Per Order', 'Profit Margin %']].describe()


Removed 0 zero/invalid-value order records (0 rows)

Clean dataset: 180,519 rows x 42 columns


,Sales,Benefit per order,Order Profit Per Order,Profit Margin %
count,"180,519.00","180,519.00","180,519.00","180,519.00"
mean,203.77,21.97,21.97,10.83
std,132.27,104.43,104.43,42.06
min,9.99,"-4,274.98","-4,274.98",-275.00
25%,119.98,7.00,7.00,6.22
50%,199.92,31.52,31.52,24.25
75%,299.95,64.80,64.80,33.60
max,"1,999.99",911.80,911.80,50.04


## 3. Revenue & Profit Overview

Compute headline revenue and profit figures, then compare trend and concentration patterns.


In [5]:
total_revenue = df['Sales'].sum()
total_profit = df['Benefit per order'].sum()
overall_margin = total_profit / total_revenue * 100
total_orders = df['Order Item Total'].count()
avg_order_value = df['Sales per customer'].mean()

print(f"Total Revenue        : ${total_revenue:,.2f}")
print(f"Total Profit         : ${total_profit:,.2f}")
print(f"Overall Profit Margin: {overall_margin:.2f}%")
print(f"Total Order Items    : {total_orders:,}")
print(f"Unique Customers     : {df['Customer Id'].nunique():,}")
print(f"Unique Products      : {df['Product Name'].nunique():,}")


Total Revenue        : $36,784,734.31
Total Profit         : $3,966,902.97
Overall Profit Margin: 10.78%
Total Order Items    : 180,519
Unique Customers     : 20,652
Unique Products      : 118


In [6]:
# Revenue vs Profit by Shipping Mode (proxy trend view - dataset has no order date column)
ship_perf = df.groupby('Shipping Mode').agg(
    Revenue=('Sales', 'sum'),
    Profit=('Benefit per order', 'sum'),
    Orders=('Order Item Total', 'count')
).reset_index()
ship_perf['Margin %'] = ship_perf['Profit'] / ship_perf['Revenue'] * 100
ship_perf = ship_perf.sort_values('Revenue', ascending=False)

fig = go.Figure()
fig.add_bar(x=ship_perf['Shipping Mode'], y=ship_perf['Revenue'], name='Revenue', marker_color=PALETTE[1])
fig.add_bar(x=ship_perf['Shipping Mode'], y=ship_perf['Profit'], name='Profit', marker_color=PALETTE[4])
fig.update_layout(barmode='group', title='Revenue vs Profit by Shipping Mode', height=420)
fig.show()
ship_perf


,Shipping Mode,Revenue,Profit,Orders,Margin %
3,Standard Class,"22,022,391.46","2,370,454.45",107752,10.76
2,Second Class,"7,145,444.68","750,308.17",35216,10.50
0,First Class,"5,674,369.65","643,121.92",27814,11.33
1,Same Day,"1,942,528.52","203,018.43",9737,10.45


In [7]:
# Profit concentration: cumulative share of profit by customer (Pareto view)
cust_profit = df.groupby('Customer Id')['Benefit per order'].sum().sort_values(ascending=False).reset_index()
cust_profit['cum_profit_share'] = cust_profit['Benefit per order'].cumsum() / cust_profit['Benefit per order'].sum() * 100
cust_profit['cust_rank_share'] = (np.arange(1, len(cust_profit)+1) / len(cust_profit)) * 100

top10pct_cutoff = int(len(cust_profit) * 0.10)
share_from_top10pct = cust_profit.iloc[:top10pct_cutoff]['Benefit per order'].sum() / total_profit * 100
print(f"Top 10% of customers ({top10pct_cutoff:,} customers) generate {share_from_top10pct:.1f}% of total profit")

fig = px.line(cust_profit, x='cust_rank_share', y='cum_profit_share',
              title='Profit Concentration Curve (Pareto) — Customers Ranked by Profit',
              labels={'cust_rank_share': '% of Customers', 'cum_profit_share': 'Cumulative % of Profit'})
fig.update_traces(line_color=PALETTE[0], line_width=3)
fig.add_shape(type='line', x0=0, y0=0, x1=100, y1=100, line=dict(dash='dot', color='#999'))
fig.update_layout(height=420)
fig.show()


Top 10% of customers (2,065 customers) generate 49.1% of total profit


## 4. Product & Category Profitability Analysis

Analyze margin by Product Name and Category Name; flag high-revenue/low-margin products and any loss-making categories.


In [8]:
cat_perf = df.groupby('Category Name').agg(
    Revenue=('Sales', 'sum'),
    Profit=('Benefit per order', 'sum'),
    Orders=('Order Item Total', 'count')
).reset_index()
cat_perf['Margin %'] = cat_perf['Profit'] / cat_perf['Revenue'] * 100
cat_perf = cat_perf.sort_values('Revenue', ascending=False)

fig = px.bar(cat_perf.head(15), x='Revenue', y='Category Name', orientation='h',
             color='Margin %', color_continuous_scale=['#D64545', '#B7D8FB', '#0B3D91'],
             title='Top 15 Categories: Revenue Sized, Margin Colored')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=520)
fig.show()

loss_making = cat_perf[cat_perf['Profit'] < 0]
print(f"Loss-making categories: {len(loss_making)}")
low_margin = cat_perf[cat_perf['Margin %'] < cat_perf['Margin %'].median()].sort_values('Revenue', ascending=False)
print("\nHigh-revenue but below-median-margin categories (watch list):")
low_margin.head(8)


Loss-making categories: 0

High-revenue but below-median-margin categories (watch list):


,Category Name,Revenue,Profit,Orders,Margin %
18,Fishing,"6,929,653.50","756,220.76",17325,10.91
9,Camping & Hiking,"4,118,425.42","427,455.57",13729,10.38
10,Cardio Equipment,"3,694,843.20","383,011.10",12487,10.37
46,Water Sports,"3,113,844.60","325,146.96",15540,10.44
34,Men's Footwear,"2,891,757.54","311,902.82",22246,10.79
30,Indoor/Outdoor Games,"2,888,993.94","318,451.43",19298,11.02
38,Shop By Sport,"1,309,522.02","129,813.96",10984,9.91
13,Computers,"663,000.00","69,656.81",442,10.51


In [9]:
prod_perf = df.groupby('Product Name').agg(
    Revenue=('Sales', 'sum'),
    Profit=('Benefit per order', 'sum'),
    Orders=('Order Item Total', 'count')
).reset_index()
prod_perf['Margin %'] = prod_perf['Profit'] / prod_perf['Revenue'] * 100

print("Top 10 products by revenue:")
display(prod_perf.sort_values('Revenue', ascending=False).head(10))

print("\nHigh-revenue, low-margin products (bottom-quartile margin among top-revenue quartile):")
top_rev_q = prod_perf['Revenue'].quantile(0.75)
low_margin_q = prod_perf['Margin %'].quantile(0.25)
flagged = prod_perf[(prod_perf['Revenue'] >= top_rev_q) & (prod_perf['Margin %'] <= low_margin_q)]
flagged.sort_values('Revenue', ascending=False)


Top 10 products by revenue:

,Product Name,Revenue,Profit,Orders,Margin %
24,Field & Stream Sportsman 16 Gun Fire Safe,"6,929,653.50","756,220.76",17325,10.91
71,Perfect Fitness Perfect Rip Deck,"4,421,143.02","493,828.30",24515,11.17
21,Diamondback Women's Serene Classic Comfort Bi,"4,118,425.42","427,455.57",13729,10.38
61,Nike Men's Free 5.0+ Running Shoe,"3,667,633.20","379,915.82",12169,10.36
59,Nike Men's Dri-FIT Victory Golf Polo,"3,147,800.00","350,421.03",21035,11.13
70,Pelican Sunstream 100 Kayak,"3,099,845.00","324,076.37",15500,10.45
56,Nike Men's CJ Elite 2 TD Football Cleat,"2,891,757.54","311,902.82",22246,10.79
67,O'Brien Men's Neoprene Life Vest,"2,888,993.94","318,451.43",19298,11.02
102,Under Armour Girls' Toddler Spine Surge Runni,"1,269,082.65","126,278.51",10617,9.95
18,Dell Laptop,"663,000.00","69,656.81",442,10.51



High-revenue, low-margin products (bottom-quartile margin among top-revenue quartile):


,Product Name,Revenue,Profit,Orders,Margin %
17,DVDs,"79,395.54","6,655.43",483,8.38
48,Men's gala suit,"43,856.80","2,006.04",208,4.57
0,Adult dog supplies,"41,524.80","3,589.26",492,8.64


In [10]:
fig = px.scatter(prod_perf, x='Revenue', y='Margin %', size='Orders', color='Margin %',
                  color_continuous_scale=['#D64545', '#B7D8FB', '#0B3D91'],
                  hover_name='Product Name',
                  title='Product Positioning: Revenue vs Profit Margin')
fig.add_hline(y=0, line_dash='dot', line_color='#666')
fig.update_layout(height=480)
fig.show()


## 5. Customer Contribution Analysis

Aggregate sales and profit by Customer Id, identify high-value vs low-margin/loss-making customers, and segment into value tiers.


In [11]:
cust_perf = df.groupby('Customer Id').agg(
    Revenue=('Sales', 'sum'),
    Profit=('Benefit per order', 'sum'),
    Orders=('Order Item Total', 'count'),
    Segment=('Customer Segment', 'first')
).reset_index()
cust_perf['Margin %'] = cust_perf['Profit'] / cust_perf['Revenue'] * 100

print("Top 10 customers by profit:")
display(cust_perf.sort_values('Profit', ascending=False).head(10))

print("\nBottom 10 customers by profit (lowest-margin / loss-making):")
cust_perf.sort_values('Profit', ascending=True).head(10)


Top 10 customers by profit:


,Customer Id,Revenue,Profit,Orders,Segment,Margin %
2612,2641,"9,130.92","2,441.97",43,Consumer,26.74
1636,1657,"9,223.71","2,196.92",42,Consumer,23.82
9755,9833,"6,059.38","1,938.39",24,Consumer,31.99
2597,2626,"6,274.36","1,928.57",27,Consumer,30.74
4958,5004,"8,164.70","1,917.99",45,Home Office,23.49
3697,3735,"6,019.33","1,906.36",27,Corporate,31.67
741,749,"7,649.38","1,855.15",38,Consumer,24.25
5509,5560,"6,528.21","1,831.46",29,Corporate,28.05
10875,10967,"5,734.41","1,822.33",27,Consumer,31.78
5007,5053,"7,411.32","1,813.34",33,Corporate,24.47



Bottom 10 customers by profit (lowest-margin / loss-making):


,Customer Id,Revenue,Profit,Orders,Segment,Margin %
1410,1428,"3,883.67","-3,868.56",13,Consumer,-99.61
13980,14086,"1,500.00","-3,442.50",1,Home Office,-229.50
18003,18109,"1,500.00","-3,366.00",1,Corporate,-224.40
14207,14313,"1,500.00","-3,000.00",1,Consumer,-200.00
17955,18061,"1,500.00","-2,592.00",1,Consumer,-172.80
14007,14113,"1,500.00","-2,550.00",1,Consumer,-170.00
13984,14090,"1,500.00","-2,351.25",1,Corporate,-156.75
14292,14398,"1,500.00","-2,328.00",1,Home Office,-155.20
14200,14306,"1,500.00","-2,280.00",1,Home Office,-152.00
14130,14236,"1,500.00","-2,255.25",1,Consumer,-150.35


In [12]:
# Value tier segmentation via profit quartiles
cust_perf['Value Tier'] = pd.qcut(cust_perf['Profit'], 4, labels=['Bronze (Q1)', 'Silver (Q2)', 'Gold (Q3)', 'Platinum (Q4)'])

tier_summary = cust_perf.groupby('Value Tier', observed=True).agg(
    Customers=('Customer Id', 'count'),
    Revenue=('Revenue', 'sum'),
    Profit=('Profit', 'sum')
).reset_index()
tier_summary['Avg Margin %'] = tier_summary['Profit'] / tier_summary['Revenue'] * 100

fig = px.bar(tier_summary, x='Value Tier', y='Profit', color='Value Tier',
             color_discrete_sequence=PALETTE, title='Total Profit Contribution by Customer Value Tier')
fig.update_layout(showlegend=False, height=420)
fig.show()
tier_summary


,Value Tier,Customers,Revenue,Profit,Avg Margin %
0,Bronze (Q1),5163,"7,236,946.40","-1,077,122.61",-14.88
1,Silver (Q2),5165,"2,895,952.27","251,273.72",8.68
2,Gold (Q3),5161,"8,949,535.70","1,201,134.77",13.42
3,Platinum (Q4),5163,"17,702,299.94","3,591,617.09",20.29


In [13]:
seg_perf = df.groupby('Customer Segment').agg(
    Revenue=('Sales', 'sum'), Profit=('Benefit per order', 'sum'), Customers=('Customer Id', 'nunique')
).reset_index()
seg_perf['Margin %'] = seg_perf['Profit'] / seg_perf['Revenue'] * 100

fig = px.pie(seg_perf, names='Customer Segment', values='Revenue', hole=0.55,
             color_discrete_sequence=PALETTE, title='Revenue Share by Customer Segment')
fig.show()
seg_perf


,Customer Segment,Revenue,Profit,Customers,Margin %
0,Consumer,"19,095,789.79","2,073,487.67",10695,10.86
1,Corporate,"11,168,406.63","1,202,574.96",6239,10.77
2,Home Office,"6,520,537.89","690,840.34",3718,10.59


## 6. Discount Impact Diagnostics

Compare margins with/without discounting, analyze discount rate vs profit ratio, and identify the discount threshold where margin erosion accelerates.


In [14]:
# Bucket discount rates and inspect resulting profit ratio
df['Discount Bucket'] = pd.cut(df['Order Item Discount Rate'],
                                bins=[-0.001, 0.0, 0.05, 0.10, 0.15, 0.20, 1.0],
                                labels=['0%', '0-5%', '5-10%', '10-15%', '15-20%', '20%+'])

disc_perf = df.groupby('Discount Bucket', observed=True).agg(
    Orders=('Order Item Total', 'count'),
    AvgProfitRatio=('Order Item Profit Ratio', 'mean'),
    Revenue=('Sales', 'sum'),
    Profit=('Benefit per order', 'sum')
).reset_index()
disc_perf['Margin %'] = disc_perf['Profit'] / disc_perf['Revenue'] * 100

fig = px.bar(disc_perf, x='Discount Bucket', y='Margin %', color='Margin %',
             color_continuous_scale=['#D64545', '#B7D8FB', '#0B3D91'],
             title='Profit Margin by Discount Rate Bucket — Erosion Threshold View')
fig.update_layout(height=420)
fig.show()
disc_perf


,Discount Bucket,Orders,AvgProfitRatio,Revenue,Profit,Margin %
0,0%,10028,0.13,"2,042,369.14","267,412.40",13.09
1,0-5%,50143,0.12,"10,215,745.61","1,171,682.08",11.47
2,5-10%,40116,0.12,"8,173,852.12","926,511.63",11.34
3,10-15%,30087,0.12,"6,131,787.37","625,339.12",10.20
4,15-20%,40116,0.12,"8,176,657.73","784,061.02",9.59
5,20%+,10029,0.13,"2,044,322.34","191,896.72",9.39


In [15]:
corr = df[['Order Item Discount Rate', 'Order Item Profit Ratio']].corr().iloc[0, 1]
print(f"Correlation between Discount Rate and Profit Ratio: {corr:.3f}")

fig = px.scatter(df.sample(min(8000, len(df)), random_state=42),
                  x='Order Item Discount Rate', y='Order Item Profit Ratio',
                  trendline='ols', opacity=0.35,
                  color_discrete_sequence=[PALETTE[1]],
                  title='Discount Rate vs Profit Ratio (sampled orders + trend line)')
fig.update_layout(height=460)
fig.show()


Correlation between Discount Rate and Profit Ratio: -0.003


## 7. Market & Regional Profit Analysis

Compare profitability across Markets, Order Regions, and Countries; flag high-revenue/weak-profit markets.


In [16]:
market_perf = df.groupby('Market').agg(
    Revenue=('Sales', 'sum'), Profit=('Benefit per order', 'sum'), Orders=('Order Item Total', 'count')
).reset_index()
market_perf['Margin %'] = market_perf['Profit'] / market_perf['Revenue'] * 100
market_perf = market_perf.sort_values('Revenue', ascending=False)

fig = go.Figure()
fig.add_bar(x=market_perf['Market'], y=market_perf['Revenue'], name='Revenue', marker_color=PALETTE[1], yaxis='y')
fig.add_trace(go.Scatter(x=market_perf['Market'], y=market_perf['Margin %'], name='Margin %',
                          yaxis='y2', mode='lines+markers', line=dict(color=NEG_COLOR, width=3)))
fig.update_layout(
    title='Market Revenue vs Profit Margin',
    yaxis=dict(title='Revenue ($)'),
    yaxis2=dict(title='Margin %', overlaying='y', side='right'),
    height=460
)
fig.show()
market_perf


,Market,Revenue,Profit,Orders,Margin %
1,Europe,"10,872,396.60","1,169,442.96",50252,10.76
2,LATAM,"10,277,612.64","1,123,321.61",51594,10.93
3,Pacific Asia,"8,273,743.58","857,753.44",41260,10.37
4,USCA,"5,066,528.61","564,313.78",25799,11.14
0,Africa,"2,294,452.88","252,071.18",11614,10.99


In [17]:
region_perf = df.groupby('Order Region').agg(
    Revenue=('Sales', 'sum'), Profit=('Benefit per order', 'sum')
).reset_index()
region_perf['Margin %'] = region_perf['Profit'] / region_perf['Revenue'] * 100
region_perf = region_perf.sort_values('Revenue', ascending=False)

fig = px.bar(region_perf, x='Order Region', y='Revenue', color='Margin %',
             color_continuous_scale=['#D64545', '#B7D8FB', '#0B3D91'],
             title='Revenue by Order Region, Colored by Margin %')
fig.update_layout(xaxis={'categoryorder': 'total descending'}, height=460)
fig.show()

print("Countries: revenue-strong but margin-weak (bottom-quartile margin, top-half revenue):")
country_perf = df.groupby('Order Country').agg(Revenue=('Sales', 'sum'), Profit=('Benefit per order', 'sum')).reset_index()
country_perf['Margin %'] = country_perf['Profit'] / country_perf['Revenue'] * 100
rev_median = country_perf['Revenue'].median()
margin_q1 = country_perf['Margin %'].quantile(0.25)
country_perf[(country_perf['Revenue'] >= rev_median) & (country_perf['Margin %'] <= margin_q1)].sort_values('Revenue', ascending=False).head(10)


Countries: revenue-strong but margin-weak (bottom-quartile margin, top-half revenue):


,Order Country,Revenue,Profit,Margin %
38,Cuba,"709,339.41","60,624.03",8.55
59,Guatemala,"550,517.11","47,330.88",8.60
129,Rusia,"200,161.53","17,473.00",8.73
4,Arabia Saudí,"163,149.70","11,735.63",7.19
77,Japón,"154,352.13","12,071.53",7.82
72,Irlanda,"116,683.22","6,588.69",5.65
40,Ecuador,"59,389.69","2,794.71",4.71
3,Angola,"57,666.02",361.80,0.63
162,Zambia,"55,521.24","4,120.77",7.42
80,Kenia,"50,662.48","3,826.89",7.55


## 8. Key Performance Indicator (KPI) Summary


In [18]:
kpi_summary = pd.DataFrame({
    'KPI': ['Total Revenue', 'Total Profit', 'Profit Margin (%)', 'Avg Sales per Customer',
            'Customer Value Index (Avg Profit/Customer)', 'Best Category Margin (%)',
            'Worst Category Margin (%)', 'Late Delivery Risk Rate (%)'],
    'Value': [
        f"${total_revenue:,.0f}",
        f"${total_profit:,.0f}",
        f"{overall_margin:.2f}%",
        f"${df['Sales per customer'].mean():,.2f}",
        f"${cust_perf['Profit'].mean():,.2f}",
        f"{cat_perf['Margin %'].max():.2f}%",
        f"{cat_perf['Margin %'].min():.2f}%",
        f"{df['Late_delivery_risk'].mean()*100:.1f}%"
    ]
})
kpi_summary


,KPI,Value
0,Total Revenue,"$36,784,734"
1,Total Profit,"$3,966,903"
2,Profit Margin (%),10.78%
3,Avg Sales per Customer,$183.11
4,Customer Value Index (Avg Profit/Customer),$192.08
5,Best Category Margin (%),17.46%
6,Worst Category Margin (%),0.61%
7,Late Delivery Risk Rate (%),54.8%


In [19]:
# Persist cleaned dataset + key aggregates for the Streamlit dashboard
out_dir = Path('../data')
df.to_parquet(out_dir / 'apl_clean.parquet', index=False)
df.to_csv(out_dir / 'apl_clean.csv', index=False)
print("Saved cleaned dataset to data/apl_clean.parquet and data/apl_clean.csv")
print(f"Final shape: {df.shape}")


Saved cleaned dataset to data/apl_clean.parquet and data/apl_clean.csv
Final shape: (180519, 43)


## 9. Key Insights & Recommendations

**Profitability is heavily concentrated.** A small share of top customers by profit account for a disproportionate share of total profit — the Pareto curve above quantifies this concentration precisely for this dataset. Retention and account-management investment should prioritize the Platinum/Gold value tiers identified in the customer segmentation.

**Revenue leadership does not equal margin leadership.** The top-revenue categories (Fishing, Cleats, Camping & Hiking, Cardio Equipment) are not necessarily the highest-margin ones. The revenue-vs-margin scatter and the "high-revenue but below-median-margin" watch list should guide pricing review, not just sales volume targets.

**Discounting shows a measurable margin cost.** The discount-rate vs profit-ratio relationship is negative — as discount buckets increase, average margin declines. The bucketed view highlights the rate at which erosion accelerates and gives a data-backed ceiling for discount policy.

**Markets and regions carry different risk/reward profiles.** Some markets post strong revenue with comparatively thinner margins; regional/country tables above flag the specific markets and countries that are revenue-heavy but margin-light, which is where commercial terms (freight allocation, discount caps, contract pricing) should be renegotiated first.

**Recommended next actions for leadership:**
1. Build account plans for Platinum/Gold-tier customers; investigate root causes for Bronze-tier / loss-making accounts (are they early-stage, discount-heavy, or high-return?).
2. Review pricing and discount policy for the flagged high-revenue/low-margin products and categories.
3. Cap or restructure discounts above the erosion threshold identified in the discount diagnostics.
4. Prioritize commercial-term renegotiation in the revenue-strong, margin-weak markets/countries.

These findings, and the interactive filters to explore them by segment/category/market/discount, are available in the companion Streamlit dashboard: `dashboard/app.py`.
